In [1]:
import os
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.Seq import Seq
# --------------------------------
from scipy.stats import mannwhitneyu, ttest_ind, pearsonr
from scipy.cluster.hierarchy import linkage, leaves_list
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import ols
# --------------------------------
from tqdm.notebook import tqdm
# ----------------------------------------------------------------
from lets_plot import *
LetsPlot.setup_html()
base_size = 14
tsne_plot_width = 650
theme_settings = theme(
    axis_text=element_text(size=base_size),
    axis_title=element_text(size=base_size * 1.2),
    legend_title=element_text(size=base_size * 1.2),
    legend_text=element_text(size=base_size * 0.9),
    plot_title=element_text(size=base_size * 1.4),
    axis_ticks_y=element_line(color='black', size=0.5),
    axis_line_y=element_line(color='black', size=0.5)
)

In [25]:
# quick fasta loader(s) ------------------
from typing import Optional, Dict
def load_fasta_to_dict(
    fasta_path: str,
    ids_keep: Optional[set] = None,
    id_func=lambda x: x,
) -> Dict[str, str]:
    """Reads FASTA into {id: sequence}. Applies id_func to header token."""
    seqs: Dict[str, str] = {}
    with open(fasta_path, 'r') as fh:
        cur_id, cur_seq = None, []
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if cur_id is not None:
                    seq = ''.join(cur_seq).upper()
                    if ids_keep is None or cur_id in ids_keep:
                        seqs[cur_id] = seq
                cur_id = id_func(line[1:].split()[0])
                cur_seq = []
            else:
                cur_seq.append(line)
        if cur_id is not None:
            seq = ''.join(cur_seq).upper()
            if ids_keep is None or cur_id in ids_keep:
                seqs[cur_id] = seq
    return seqs
## solely to load the RNAfold output
import re
def parse_dotbracket_file(path):
    records = []
    with open(path) as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(0, len(lines), 3):
        header = lines[i]
        seq = lines[i+1]
        struct_line = lines[i+2]

        seq_id = header.lstrip(">")

        m = re.search(r"\((-?\d+(?:\.\d+)?)\)$", struct_line)
        energy = float(m.group(1)) if m else None

        records.append({"id": seq_id,
                        "sequence": seq,
                        "structure": struct_line,
                        "MFE": energy})
    return pd.DataFrame(records)

In [26]:
# functions to estimate motif enrichment via GLM

## ---------------------------------
import re, math, numpy as np, pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# G4 motifs per AM
motifs = {
    "G4_3t_short": r'G{3,}[ACGT]{1,7}G{3,}[ACGT]{1,7}G{3,}[ACGT]{1,7}G{3,}',
    "G4_3t_long":  r'G{3,}[ACGT]{8,12}G{3,}[ACGT]{8,12}G{3,}[ACGT]{8,12}G{3,}',
    "G4_2t_short": r'G{2,}[ACGT]{1,7}G{2,}[ACGT]{1,7}G{2,}[ACGT]{1,7}G{2,}',
    "G4_2t_long":  r'G{2,}[ACGT]{8,12}G{2,}[ACGT]{8,12}G{2,}[ACGT]{8,12}G{2,}',
}


def compile_patterns(name_to_regex, overlap=True, ignore_case=True):
    flags = re.IGNORECASE if ignore_case else 0
    if overlap:
        # wrap with lookahead to capture overlapping matches
        comp = {name: re.compile(rf'(?=({pat}))', flags) for name, pat in name_to_regex.items()}
    else:
        comp = {name: re.compile(pat, flags) for name, pat in name_to_regex.items()}
    return comp

def iter_matches(seq, cregex):
    for m in cregex.finditer(seq):
        s = m.start(1) if m.lastindex else m.start()
        e = s + len(m.group(1) if m.lastindex else m.group(0))
        yield s, e

def gc_fraction(seq):
    s = seq.upper()
    if not s: return 0.0
    g = s.count('G'); c = s.count('C')
    return (g + c) / len(s)

# ---------- Build per-sequence feature table ----------
def summarize_sequences(seq_dict, group_map, compiled_patterns):
    rows = []
    for sid, seq in seq_dict.items():
        L = len(seq)
        if L == 0: continue
        gcf = gc_fraction(seq)
        grp = group_map[sid] if group_map is not None else "NA"
        for name, crex in compiled_patterns.items():
            # count overlapping hits
            cnt = sum(1 for _ in iter_matches(seq, crex))
            rows.append(dict(sequence_id=sid, group=grp, motif=name, length=L, gc=gcf, count=cnt))
    return pd.DataFrame(rows)

# ---------- Fit GLM: count ~ group with loglen offset ----------
def fit_glm_enrichment(df, family="poisson"):
    out = []
    for motif, sub in df.groupby("motif", sort=False):
        sub = sub.copy()
        sub["log_len"] = np.log(sub["length"])
        formula = f"count ~ C(group)"
        if family == "poisson":
            model = smf.glm(formula, data=sub, family=sm.families.Poisson(), offset=sub["log_len"])
            res = model.fit(cov_type="HC3")
        elif family == "nb":
            model = smf.glm(formula, data=sub, family=sm.families.NegativeBinomial(), offset=sub["log_len"])
            res = model.fit()
        else:
            raise ValueError("family must be 'poisson' or 'nb'")

        group_counts = sub.groupby("group").size().to_dict()
        n_total = int(len(sub))

        # report rate ratios
        for p, est in res.params.items():
            if p.startswith("C(group)[T."):
                term_group = p.split("C(group)[T.", 1)[1][:-1]
                n_term = int(group_counts.get(term_group, 0))
                ref_group = np.nan
                n_ref = np.nan
                if len(group_counts) == 2 and term_group in group_counts:
                    ref_group = [g for g in group_counts if g != term_group][0]
                    n_ref = int(group_counts.get(ref_group, 0))

                rr = math.exp(est)
                lo, hi = np.exp(res.conf_int().loc[p].values)
                out.append(dict(
                    motif=motif,
                    term=p,
                    group_term=term_group,
                    group_ref=ref_group,
                    n_term=n_term,
                    n_ref=n_ref,
                    n_total=n_total,
                    rate_ratio=rr,
                    ci_low=lo,
                    ci_high=hi,
                    pvalue=res.pvalues[p],
                    family=family
                ))
    return pd.DataFrame(out)

In [27]:
# functions to compute motif meta-coverage profiles

def _bin_edges(a: float, b: float, nbins: int) -> np.ndarray:
    return np.linspace(a, b, nbins + 1, dtype=float)

def _merge_intervals(starts, ends):
    if len(starts) == 0: return []
    order = np.argsort(starts)
    s = np.array(starts)[order]; e = np.array(ends)[order]
    out = []; cs, ce = s[0], e[0]
    for i in range(1, len(s)):
        if s[i] <= ce:
            ce = max(ce, e[i])
        else:
            out.append((cs, ce)); cs, ce = s[i], e[i]
    out.append((cs, ce))
    return out

def _covered_bp_per_bin(intervals, edges):
    # intervals: list of (s,e) in the region's local coords
    cov = np.zeros(len(edges)-1, dtype=float)
    if not intervals: return cov
    for (s,e) in intervals:
        s = float(s); e = float(e)
        if e <= edges[0] or s >= edges[-1]: continue
        i0 = np.searchsorted(edges, s, side='right') - 1
        i1 = np.searchsorted(edges, e, side='left')
        i0 = max(i0, 0); i1 = min(i1, len(edges)-1)
        for i in range(i0, i1):
            lo = max(edges[i], s); hi = min(edges[i+1], e)
            if hi > lo: cov[i] += (hi - lo)
    return cov

def _runs_per_bin(runs, edges):
    # runs: list[(s,e)], count if a run touches bin i
    cnt = np.zeros(len(edges)-1, dtype=float)
    for (s,e) in runs:
        i0 = np.searchsorted(edges, s, side='right') - 1
        i1 = np.searchsorted(edges, e, side='left')
        i0 = max(i0, 0); i1 = min(i1, len(edges)-1)
        for i in range(i0, i1):
            cnt[i] += 1.0
    return cnt

# ---------- Build a hits table from regex scanning ----------
def scan_regex_hits(seq_dict, compiled_patterns, group_map=None):
    """
    Returns a tidy DataFrame: sequence_id, motif, run_start, run_length, seq_len, group
    """
    recs = []
    for sid, seq in seq_dict.items():
        L = len(seq)
        grp = group_map[sid] if group_map is not None else "NA"
        for motif, crex in compiled_patterns.items():
            for s,e in iter_matches(seq, crex):
                recs.append(dict(sequence_id=sid, motif=motif,
                                 run_start=s, run_length=(e-s), seq_len=L, group=grp))
    df = pd.DataFrame(recs)
    if df.empty:
        return pd.DataFrame(columns=["sequence_id","motif","run_start","run_length","seq_len","group"])
    return df

# ---------- Meta-coverage for regex motifs ----------
def metacoverage_regex(
    hits_df,
    seq_lookup,                # dict or callable id->sequence
    group_col='group',
    label_col='motif',
    flank_len=50,
    nbins_left=None,
    nbins_core=100,
    nbins_right=None,
    per_seq_mode='any',        # 'any'|'count'|'length'|'frac'|'binary_fraction'
    label_order=None,
    group_order=None
):
    if nbins_left  is None: nbins_left  = flank_len
    if nbins_right is None: nbins_right = flank_len

    total_len = nbins_left + nbins_core + nbins_right
    off_left, off_core, off_right = 0, nbins_left, nbins_left + nbins_core
    labels = hits_df[label_col].unique() if label_order is None else list(label_order)
    groups = hits_df[group_col].unique() if group_order is None else list(group_order)

    use_union_bins = per_seq_mode in ('frac', 'binary_fraction')
    frames = []

    for g in groups:
        sub_g = hits_df[hits_df[group_col] == g]
        if sub_g.empty:
            continue
        nseqs_group = max(1, sub_g['sequence_id'].nunique())

        for lab in labels:
            sub_lab = sub_g[sub_g[label_col] == lab]
            acc = np.zeros(total_len, dtype=float)

            # iterate sequences to give each sequence equal weight
            for sid, gseq in sub_lab.groupby('sequence_id', sort=False):
                # pull sequence length (from table) and/or actual string for 'frac'
                L = int(gseq['seq_len'].iloc[0])
                s = gseq['run_start'].to_numpy(np.int64, copy=False)
                rl = gseq['run_length'].to_numpy(np.int64, copy=False)
                e = s + rl

                seq_cov = np.zeros(total_len, dtype=float)

                # 'frac' denominator: #nt in each bin (computed from actual sequence if available)
                denom = None
                if per_seq_mode == 'frac':
                    seq_str = seq_lookup(sid) if callable(seq_lookup) else seq_lookup[sid]
                    seq_b = np.frombuffer(seq_str.encode('ascii'), dtype=np.uint8)

                    # For 'frac', denominator is simply bin length in nt (independent of nucleotide),
                    # because regex hits are not nucleotide-specific like polyN.
                    bin_len = np.concatenate([
                        np.full(nbins_left,  flank_len, dtype=float) if nbins_left  > 0 else np.zeros(0),
                        np.full(nbins_core,  max(1, L - 2*flank_len), dtype=float) if nbins_core  > 0 else np.zeros(0),
                        np.full(nbins_right, flank_len, dtype=float) if nbins_right > 0 else np.zeros(0),
                    ])
                    denom = bin_len

                # LEFT region [0, flank_len)
                if flank_len > 0 and nbins_left > 0:
                    ls = np.clip(s, 0, flank_len); le = np.clip(e, 0, flank_len)
                    valid = le > ls
                    if np.any(valid):
                        intervals = _merge_intervals(ls[valid], le[valid]) if use_union_bins else list(zip(ls[valid], le[valid]))
                        edges = _bin_edges(0.0, float(flank_len), nbins_left)
                        if per_seq_mode == 'any':
                            # any: 1 if any run touches the bin
                            touch = _runs_per_bin(intervals if not use_union_bins else intervals, edges)
                            seq_cov[off_left:off_left+nbins_left] = (touch > 0).astype(float)
                        elif per_seq_mode == 'count':
                            seq_cov[off_left:off_left+nbins_left] = _runs_per_bin(intervals, edges)
                        elif per_seq_mode == 'length':
                            seq_cov[off_left:off_left+nbins_left] = _covered_bp_per_bin(intervals if use_union_bins else list(zip(ls[valid], le[valid])), edges)
                        elif per_seq_mode in ('frac','binary_fraction'):
                            vals = _covered_bp_per_bin(intervals, edges) if per_seq_mode=='frac' else _runs_per_bin(intervals, edges)
                            seq_cov[off_left:off_left+nbins_left] = vals

                # RIGHT region [L-50, L)
                if flank_len > 0 and nbins_right > 0:
                    right0 = L - flank_len
                    rs = np.clip(s, right0, L) - right0
                    re = np.clip(e, right0, L) - right0
                    valid = re > rs
                    if np.any(valid):
                        intervals = _merge_intervals(rs[valid], re[valid]) if use_union_bins else list(zip(rs[valid], re[valid]))
                        edges = _bin_edges(0.0, float(flank_len), nbins_right)
                        if per_seq_mode == 'any':
                            touch = _runs_per_bin(intervals if not use_union_bins else intervals, edges)
                            seq_cov[off_right:off_right+nbins_right] = (touch > 0).astype(float)
                        elif per_seq_mode == 'count':
                            seq_cov[off_right:off_right+nbins_right] = _runs_per_bin(intervals, edges)
                        elif per_seq_mode == 'length':
                            seq_cov[off_right:off_right+nbins_right] = _covered_bp_per_bin(intervals if use_union_bins else list(zip(rs[valid], re[valid])), edges)
                        elif per_seq_mode in ('frac','binary_fraction'):
                            vals = _covered_bp_per_bin(intervals, edges) if per_seq_mode=='frac' else _runs_per_bin(intervals, edges)
                            seq_cov[off_right:off_right+nbins_right] = vals

                # CORE region [50, L-50) rescaled
                core0, core1 = flank_len, L - flank_len
                core_len_abs = max(0, core1 - core0)
                if core_len_abs > 0 and nbins_core > 0:
                    cs = np.clip(s, core0, core1) - core0
                    ce = np.clip(e, core0, core1) - core0
                    valid = ce > cs
                    if np.any(valid):
                        intervals = _merge_intervals(cs[valid], ce[valid]) if use_union_bins else list(zip(cs[valid], ce[valid]))
                        edges = _bin_edges(0.0, float(core_len_abs), nbins_core)
                        if per_seq_mode == 'any':
                            touch = _runs_per_bin(intervals if not use_union_bins else intervals, edges)
                            seq_cov[off_core:off_core+nbins_core] = (touch > 0).astype(float)
                        elif per_seq_mode == 'count':
                            seq_cov[off_core:off_core+nbins_core] = _runs_per_bin(intervals, edges)
                        elif per_seq_mode == 'length':
                            seq_cov[off_core:off_core+nbins_core] = _covered_bp_per_bin(intervals if use_union_bins else list(zip(cs[valid], ce[valid])), edges)
                        elif per_seq_mode in ('frac','binary_fraction'):
                            vals = _covered_bp_per_bin(intervals, edges) if per_seq_mode=='frac' else _runs_per_bin(intervals, edges)
                            seq_cov[off_core:off_core+nbins_core] = vals

                # normalise for 'frac'
                if per_seq_mode == 'frac':
                    with np.errstate(divide='ignore', invalid='ignore'):
                        seq_cov = np.where(denom > 0, seq_cov / denom, 0.0)

                acc += seq_cov

            avg = acc / nseqs_group  # equal weight per sequence
            frames.append(pd.DataFrame({
                "bin": np.arange(total_len, dtype=int),
                "coverage": avg,
                label_col: lab,
                "group": g
            }))

    long_df = pd.concat(frames, ignore_index=True) if frames else \
              pd.DataFrame(columns=['bin','coverage',label_col,'group'])

    # annotate regions
    if not long_df.empty:
        L0, L1 = 0, nbins_left - 1
        C0, C1 = nbins_left, nbins_left + nbins_core - 1
        R0, R1 = nbins_left + nbins_core, total_len - 1
        b = long_df['bin'].to_numpy()
        region = np.where((b >= L0) & (b <= L1), 'left',
                 np.where((b >= C0) & (b <= C1), 'core', 'right'))
        long_df['region'] = pd.Categorical(region, ['left','core','right'], ordered=True)

    return long_df, dict(
        left_bins=(0, nbins_left-1),
        core_bins=(nbins_left, nbins_left+nbins_core-1),
        right_bins=(nbins_left+nbins_core, nbins_left+nbins_core+nbins_right-1),
        flank_len_bp=flank_len, nbins_left=nbins_left, nbins_core=nbins_core, nbins_right=nbins_right
    )


In [2]:
## function to prepare a contour plot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
import seaborn as sns


def plot_2d_density_contours_continuous(
        x, y, cmap_name='viridis',
        kde_smooth=1, levels=11, linewidth=2.3, 
        base_alpha=1, min_alpha=.5,
        alpha_decay='linear', decay_rate=2.0
    ):
    xy = np.vstack([x, y])
    kde = gaussian_kde(xy)
    kde.set_bandwidth(bw_method=kde.factor * kde_smooth)  # smoother KDE

    ##
    x_margin = (x.max() - x.min()) * 0.1
    y_margin = (y.max() - y.min()) * 0.1
    xmin, xmax = x.min() - x_margin, x.max() + x_margin
    ymin, ymax = y.min() - y_margin, y.max() + y_margin
    xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

    min_level = np.percentile(zz, 50)  # ignore outer noise
    levels_vals = np.linspace(min_level, zz.max(), levels)

    ##
    if alpha_decay == 'linear':
        alphas = np.linspace(base_alpha, min_alpha, len(levels_vals))
    elif alpha_decay == 'exponent':
        exp_range = np.linspace(0, 1, len(levels_vals))
        alphas = base_alpha * np.exp(-decay_rate * exp_range)
    cmap = plt.get_cmap(cmap_name)

    ##
    for i, lev in enumerate(levels_vals):
        plt.contour(
            xx, yy, zz,
            levels=[lev],
            colors=[cmap(i / (levels - 1))],
            linewidths=linewidth,
            alpha=alphas[::-1][i] 
        )

def plot_2dd_scatter(
        tab_data, contour_col,
        x_name, y_name,
        x_lab=None, y_lab=None,
        scat_alpha=0.2,
        contour_col_classes=None, 
        dlevels=6, linewidth=2.3, kde_smooth=1,
        base_alpha=1, min_alpha=.5,
        alpha_decay='linear', decay_rate=2.0,
        cmaps='viridis', fig_size=(8,8),
        file_name=None,
        show_plot=True
    ):
    plt.figure(figsize=fig_size)
    sns.scatterplot(
        data=tab_data, x=x_name, y=y_name,
        edgecolor=None, color='gray',
        alpha=scat_alpha, s=3, legend=False, rasterized=True
    )
    if not x_lab:
        x_lab = x_name
    if not y_lab:
        y_lab = y_name
    if contour_col_classes:
        tab_data = tab_data[tab_data[contour_col].isin(list(contour_col_classes.keys()))]
        for cls, cls_col in contour_col_classes.items():
            subset = tab_data[tab_data[contour_col] == cls]
            cls_cmap = LinearSegmentedColormap.from_list(cls, ['white', cls_col], N=dlevels)
            plot_2d_density_contours_continuous(
                subset[x_name], subset[y_name], cmap_name=cls_cmap, kde_smooth=kde_smooth, levels=dlevels,
                linewidth=linewidth, base_alpha=base_alpha, min_alpha=min_alpha,
                alpha_decay=alpha_decay, decay_rate=decay_rate
            )
    ##
    else:
        for cls in tab_data.dropna(subset=contour_col)[contour_col].unique():
            subset = tab_data[tab_data[contour_col] == cls]
            plot_2d_density_contours_continuous(subset[x_name], subset[y_name], cmap_name=cmap)

    plt.xlabel(x_lab, fontsize=10)
    plt.ylabel(y_lab, fontsize=10)
    ##
    if file_name:
        plt.savefig(f'{file_name}', format='pdf', bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close()


In [8]:
# function to compute ROC and PRC statistics

from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    auc, confusion_matrix
)
def compute_roc_prc(y_true, y_score, pos_label=1, sample_weight=None):
    """
    Compute ROC and Precision-Recall curve statistics.

    Returns a dict with:
      - fpr, tpr, thresholds_roc, roc_auc
      - precision, recall, thresholds_pr, average_precision, pr_auc
      - best_threshold_youdenJ, confusion_at_best_thr
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # ROC
    fpr, tpr, thr_roc = roc_curve(y_true, y_score,
                                  pos_label=pos_label,
                                  sample_weight=sample_weight)
    roc_auc = roc_auc_score(y_true, y_score,
                            sample_weight=sample_weight)

    # PRC
    precision, recall, thr_pr = precision_recall_curve(
        y_true, y_score,
        pos_label=pos_label,
        sample_weight=sample_weight
    )
    ap = average_precision_score(y_true, y_score,
                                 sample_weight=sample_weight)
    pr_auc = auc(recall, precision)

    # best threshold (Youden’s J)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_thr = thr_roc[best_idx]
    yhat = (y_score >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yhat).ravel()

    return {
        "roc": {
            "fpr": fpr,
            "tpr": tpr,
            "thresholds": thr_roc,
            "roc_auc": roc_auc
        },
        "prc": {
            "precision": precision,
            "recall": recall,
            "thresholds": thr_pr,
            "average_precision": ap,
            "pr_auc": pr_auc
        },
        "best_threshold_youdenJ": float(best_thr),
        "youdenJ": float(j_scores[best_idx]),
        "confusion_at_best_thr": {"tp": int(tp), "fp": int(fp),
                                  "tn": int(tn), "fn": int(fn)}
    }

In [ ]:
## function to prepare a scatter plot
import matplotlib.pyplot as plt
import seaborn as sns

def plot_scatter_colored(tab_data, x_name, y_name, color_col,
                         x_lab=None, y_lab=None,
                         cmap='viridis', fig_size=(8, 8),
                         point_size=10, alpha=0.8,
                         vmin=None, vmax=None,
                         file_name=None, dpi=300, 
                         show_plot=True):
    """
    Plot a simple scatter plot colored by a continuous variable, with optional clipping.

    Parameters
    ----------
    tab_data : pandas.DataFrame
        Input table containing x, y, and color columns.
    x_name, y_name, color_col : str
        Column names for x-axis, y-axis, and color mapping.
    cmap : str
        Matplotlib colormap name for the continuous variable.
    vmin, vmax : float or None
        Minimum and maximum values to clip the color variable.
    """
    plt.figure(figsize=fig_size)

    # Clip color values if requested
    colors = tab_data[color_col].copy()
    if vmin is not None:
        colors = np.maximum(colors, vmin)
    if vmax is not None:
        colors = np.minimum(colors, vmax)

    sc = plt.scatter(
        tab_data[x_name],
        tab_data[y_name],
        c=colors,
        cmap=cmap,
        s=point_size,
        alpha=alpha,
        edgecolors='none',
        rasterized=True,
        vmin=vmin,
        vmax=vmax
    )

    plt.colorbar(sc, label=color_col)

    plt.xlabel(x_lab if x_lab else x_name, fontsize=10)
    plt.ylabel(y_lab if y_lab else y_name, fontsize=10)

    if file_name:
        plt.savefig(file_name, format='pdf', bbox_inches='tight', dpi=dpi)
    if show_plot:
        plt.show()
    else:
        plt.close()

In [9]:
## paths
path_metadata = os.path.join('resubmission', 'data', 'metadata_selected.csv')
dir_data = os.path.join('resubmission', 'data')
dir_results = os.path.join('resubmission', 'results')
dir_plotting = os.path.join(dir_results, 'plots')
dir_models = os.path.join(dir_results, 'models')
## load data
tab_meta = pd.read_csv(path_metadata, index_col=0)

------
-----

In [10]:
## Figure 3 B - performance of IR and HL models

val_metrics = []
val_curves_prc = []
val_curves_roc = []

for mdl in ['IR', 'HL_revised']:
    dir_model_res = os.path.join(dir_models,  mdl)
    if 'HL' in mdl:
        mdl = 'HL'
        dir_model_res = os.path.join(dir_model_res,  'parnet_512_jc_50percgap_256bp_PIR10')
    for fld in os.listdir(dir_model_res):
        if 'beststate_fold' not in fld:
            continue
        res_val = pd.read_csv(os.path.join(dir_model_res, fld, 'evaluation_results.csv'))
        n_total = len(res_val)
        n_pos = int((res_val['label'] == 1).sum())
        n_neg = int((res_val['label'] == 0).sum())
        val_auc = roc_auc_score(res_val['label'], res_val['prediction'])
        val_ap  = average_precision_score(res_val['label'], res_val['prediction'])
        val_metrics.append({
            'model': mdl,
            'fold': fld[-1],
            'n_total': n_total,
            'n_pos': n_pos,
            'n_neg': n_neg,
            'ROC': val_auc,
            'PRC': val_ap
        })
        out_tst = compute_roc_prc(res_val['label'], res_val['prediction'])
        tab_roc = pd.DataFrame(
            {
                'model': mdl,
                'fold': fld[-1],
                'tpr': out_tst['roc']['tpr'],
                'fpr': out_tst['roc']['fpr']
            }
        )
        tab_prc = pd.DataFrame(
            {
                'model': mdl,
                'fold': fld[-1],
                'precision': out_tst['prc']['precision'],
                'recall': out_tst['prc']['recall']
            }
        )
        val_curves_roc.append(tab_roc)
        val_curves_prc.append(tab_prc)
tab_metrics = pd.DataFrame(val_metrics)
tab_metrics = tab_metrics.melt(
    value_vars=['ROC', 'PRC'],
    value_name='AU',
    var_name='metrics',
    id_vars=['model', 'fold', 'n_total', 'n_pos', 'n_neg']
)
##
tab_curves_roc = pd.concat(val_curves_roc, axis=0)
tab_curves_prc = pd.concat(val_curves_prc, axis=0)

##
tab_curves_prc_ir = tab_curves_prc[tab_curves_prc.model == 'IR'].query('precision > 0').copy()
tab_curves_prc_hl = tab_curves_prc[tab_curves_prc.model == 'HL'].query('precision > 0').copy()
tab_curves_prc_plt = tab_curves_prc[tab_curves_prc.precision > 0].copy()
pos_frac_ir = tab_curves_prc_ir['precision'].min()
pos_frac_hl = tab_curves_prc_hl['precision'].min()
perf_prc = ggplot(tab_curves_prc_plt) +\
    geom_line(aes(group='fold', color='model', y='precision', x='recall'), size=.8, alpha=.7, tooltips='none') +\
    scale_color_manual(values={'IR': '#2A1F3F', 'HL': '#A76794'}) +\
    geom_hline(yintercept=pos_frac_ir, linetype='dashed', color='#2A1F3F', alpha=.7, size=1) +\
    geom_hline(yintercept=pos_frac_hl, linetype='dashed', color='#A76794', alpha=.7, size=1) +\
    theme_settings +\
    labs(color='Target:', x='Recall', y='Precision') +\
    coord_cartesian(ylim=[0, 1]) +\
    ggsize(550, 450)
ggsave(perf_prc, filename='F3B_lines_auprc_irhl.svg', path=dir_plotting)

# Source data export for Figure 3B PRC
tab_curves_prc_plt.to_csv(os.path.join(dir_plotting, 'F3B_lines_auprc_irhl_source.csv'), index=False)

##
tab_curves_roc_plt = tab_curves_roc.copy()
perf_roc = ggplot(tab_curves_roc_plt) +\
    geom_line(aes(group='fold', color='model', y='tpr', x='fpr'), size=.8, alpha=.7) +\
    geom_abline(intercept=0, slope=1, linetype='dashed', size=1) +\
    scale_color_manual(values={'IR': '#2A1F3F', 'HL': '#A76794'}) +\
    theme_settings +\
    labs(color='Target:', y='True Positive Rate', x='False Positive Rate') +\
    coord_cartesian(ylim=[0, 1]) +\
    ggsize(550, 450)
ggsave(perf_roc, filename='F3B_lines_auroc_irhl.svg', path=dir_plotting)

# Source data export for Figure 3B ROC
tab_curves_roc_plt.to_csv(os.path.join(dir_plotting, 'F3B_lines_auroc_irhl_source.csv'), index=False)

##
tab_metrics.to_csv(os.path.join(dir_plotting, 'F3B_performance_metrics.csv'), index=False)

----

In [12]:
## Figure 3 D - histogram of half-lives split into classes

high_cut, low_cut  = -0.364742, -1.342584
pir_cut = 10
tmp_hls_hist = tab_meta.query('PIR_Nuc_baseline > @pir_cut').dropna(subset=['hl_gated_intwise_scaled_logged']).copy()
class_conds = [
    tmp_hls_hist['hl_gated_intwise_scaled_logged'] > high_cut,
    tmp_hls_hist['hl_gated_intwise_scaled_logged'] < low_cut
]
class_names = ['LL-RIs', 'SL-RIs']
tmp_hls_hist['class_hls'] = np.select(class_conds, class_names, default='else')

gg_hist_hls = ggplot(data=tmp_hls_hist) +\
    geom_histogram(aes(x='hl_gated_intwise_scaled_logged', fill='class_hls', alpha='class_hls'), color='black', bins=30) +\
    geom_vline(xintercept=high_cut, color='#2a1f3f', linetype='dashed') + \
    geom_vline(xintercept=low_cut, color='#a76794', linetype='dashed') +\
    scale_fill_manual(values={'SL-RIs': '#a76794', 'LL-RIs': '#2a1f3f', 'else': 'grey'}) +\
    scale_alpha_manual(values={'SL-RIs': .7, 'LL-RIs': .7, 'else': .3}) +\
    ggsize(650, 350) +\
    labs(
        fill='Intron\nStability\ngroup:', alpha='Intron\nStability\ngroup:',
        x='log(Intron Half-life / Gene Half-life)', y='n introns'
    ) +\
    theme_settings
ggsave(gg_hist_hls, filename='F3D_hist_classSplit_hls.svg', path=dir_plotting)

# Source data export for Figure 3D
tmp_hls_hist.to_csv(os.path.join(dir_plotting, 'F3D_hist_classSplit_hls_source.csv'))

---

In [ ]:
## Figure 3 C - UMAP reduction of retained introns + class contours
tab_umap_ir = pd.read_csv(
    os.path.join(
        dir_results, 'models', 'IR',
        'umap_embeddings.csv'
    ),
    index_col=0
)
tab_ir = tab_meta.merge(
    tab_umap_ir, left_index=True, right_index=True, how='inner'
)
class_conds = [
    tab_ir['PIR_Nuc_baseline'] >= 30,
    tab_ir['PIR_Nuc_baseline'] < 1
]
class_names = ['Retained', 'Spliced']
tab_ir['class_ir'] = np.select(class_conds, class_names, default='else')

plot_2dd_scatter(
    tab_ir, 'class_ir',
    scat_alpha=0.4, kde_smooth=1.6,
    contour_col_classes = {'Spliced': '#a76794', 'Retained': '#2a1f3f'},
    x_name='UMAP1_IR', y_name='UMAP2_IR', dlevels=10,
    alpha_decay='exponent',
    show_plot=False, file_name=os.path.join(dir_plotting, 'F3E_umap_ir_contours.svg')
    )

# Source data export for Figure 3E
tab_ir.to_csv(os.path.join(dir_plotting, 'F3E_umap_ir_contours_source.csv'))

---

In [17]:
## Figure 3 E - UMAP reduction of LL-/SL- retained introns + class contours
high_cut, low_cut  = -0.364742, -1.342584
tab_umap_hlrev = pd.read_csv(
    os.path.join(
        dir_results, 'models', 'HL_revised', 'parnet_512_jc_50percgap_256bp_PIR10',
        'umap_embeddings.csv'
    ),
    index_col=0
)
tab_hl_rev = tab_meta.merge(
    tab_umap_hlrev, left_index=True, right_index=True, how='inner'
)
class_conds = [
    tab_hl_rev['hl_gated_intwise_scaled_logged'] > high_cut,
    tab_hl_rev['hl_gated_intwise_scaled_logged'] < low_cut
]
class_names = ['LL-RIs', 'SL-RIs']
tab_hl_rev['class_hls'] = np.select(class_conds, class_names, default='else')

plot_2dd_scatter(
    tab_hl_rev, 'class_hls',
    scat_alpha=0.4, kde_smooth=1.6,
    contour_col_classes = {'SL-RIs': '#a76794', 'LL-RIs': '#2a1f3f'},
    x_name='UMAP1_HLrev', y_name='UMAP2_HLrev', dlevels=10,
    alpha_decay='exponent',
    show_plot=False, file_name=os.path.join(dir_plotting, 'F3E_umap_hlrev_contours.svg')
    )

# Source data export for Figure 3E
tab_hl_rev.to_csv(os.path.join(dir_plotting, 'F3E_umap_hlrev_contours_source.csv'))

----

In [ ]:
## Figure 3 F - boxplots of delta spckle RBP peaks in LL- vs SL-RIs and Retained vs Spliced introns
# The plot generation is in 4.remake_sfig4.ipynb for data reasons

---

In [ ]:
## Figure 3 G - Boxplots showing the distribution of speckle enrichement lg2FC in IR and HL classes

tmp_hls = tab_meta.query('PIR_Nuc_baseline > @pir_cut').dropna(subset=['hl_gated_intwise_scaled_logged']).copy()
class_conds = [
    tmp_hls['hl_gated_intwise_scaled_logged'] > high_cut,
    tmp_hls['hl_gated_intwise_scaled_logged'] < low_cut
]
class_names = ['LL-RIs', 'SL-RIs']
tmp_hls['class_hls'] = np.select(class_conds, class_names, default='else')
tmp_hls = tmp_hls[tmp_hls.class_hls.isin(class_names)]

binder = []
a = tmp_hls.loc[tmp_hls['class_hls'] == 'LL-RIs', 'Speckle_Enrichment'].dropna()
b = tmp_hls.loc[tmp_hls['class_hls'] == 'SL-RIs', 'Speckle_Enrichment'].dropna()
n_ll, n_sl = len(a), len(b)
stat_t, p_t = ttest_ind(a, b, equal_var=False)
test_res_hls = pd.DataFrame(
    {
        'comparison': 'LL-RIs vs SL-RIs',
        'group_X': 'LL-RIs',
        'group_Y': 'SL-RIs',
        'n_X': n_ll,
        'n_Y': n_sl,
        'n_total': n_ll + n_sl,
        'value': 'Nuclear speckle Enrichment [lg2FC]',
        'group': '-',
        'test': ["Welch's t-test"],
        'stat': [stat_t],
        'p_value': [p_t]
    }
)

gg_speck_hls = ggplot(tmp_hls) +\
    geom_boxplot(aes(y='class_hls', x='Speckle_Enrichment', fill='class_hls', alpha='class_hls'), width=.7, outlier_size=0.7, outlier_alpha=0.35) +\
    geom_vline(xintercept=0, linetype='dashed', color='red') +\
    labs(x='Nuclear speckle Enrichment [lg2FC]', y='IR HL group') +\
    scale_fill_manual(values={'LL-RIs': '#2a1f3f', 'SL-RIs': '#a76794'}) +\
    scale_alpha_manual(values={'LL-RIs': .7, 'SL-RIs': .7}) +\
    scale_x_continuous(breaks=[-6, -4, -2, 0, 2, 4, 6]) +\
    coord_cartesian(xlim=[-6.5, 6.9]) +\
    ggsize(450, 150) +\
    theme_settings +\
    theme(legend_position='none')
ggsave(gg_speck_hls, filename='F3G_boxes_speck_hls.svg', path=dir_plotting)

# Source data export for Figure 3G (HL classes)
tmp_hls.to_csv(os.path.join(dir_plotting, 'F3G_boxes_speck_hls_source.csv'))

## ---------------------------------
tmp_ir = tab_meta.copy()
class_conds = [
    tmp_ir['PIR_Nuc_baseline'] >= 30,
    tmp_ir['PIR_Nuc_baseline'] < 1
]
class_names = ['Retained', 'Spliced']
tmp_ir['class_ir'] = np.select(class_conds, class_names, default='else')
tmp_ir = tmp_ir[tmp_ir.class_ir.isin(class_names)]

binder = []
a = tmp_ir.loc[tmp_ir['class_ir'] == 'Retained', 'Speckle_Enrichment'].dropna()
b = tmp_ir.loc[tmp_ir['class_ir'] == 'Spliced', 'Speckle_Enrichment'].dropna()
n_ret, n_spl = len(a), len(b)
stat_t, p_t = ttest_ind(a, b, equal_var=False)
test_res_ir = pd.DataFrame(
    {
        'comparison': 'Retained vs Spliced',
        'group_X': 'Retained',
        'group_Y': 'Spliced',
        'n_X': n_ret,
        'n_Y': n_spl,
        'n_total': n_ret + n_spl,
        'value': 'Nuclear speckle Enrichment [lg2FC]',
        'group': '-',
        'test': ["Welch's t-test"],
        'stat': [stat_t],
        'p_value': [p_t]
    }
)

gg_speck_ir = ggplot(tmp_ir) +\
    geom_boxplot(aes(y='class_ir', x='Speckle_Enrichment', fill='class_ir', alpha='class_ir'), width=.7, outlier_size=0.7, outlier_alpha=0.35) +\
    geom_vline(xintercept=0, linetype='dashed', color='red') +\
    labs(x='Nuclear speckle Enrichment [lg2FC]', y='IR group') +\
    scale_fill_manual(values={'Spliced': '#a76794', 'Retained': '#2a1f3f'}) +\
    scale_alpha_manual(values={'Spliced': .7, 'Retained': .7}) +\
    scale_x_continuous(breaks=[-6, -4, -2, 0, 2, 4, 6]) +\
    coord_cartesian(xlim=[-6.5, 6.9]) +\
    ggsize(450, 150) +\
    theme_settings +\
    theme(legend_position='none')
ggsave(gg_speck_ir, filename='F3G_boxes_speck_ir.svg', path=dir_plotting)

# Source data export for Figure 3G (IR classes)
tmp_ir.to_csv(os.path.join(dir_plotting, 'F3G_boxes_speck_ir_source.csv'))

## ---------------------------------
test_res = pd.concat([test_res_hls, test_res_ir], axis=0)
test_res['p_adj'] = multipletests(test_res['p_value'], method='fdr_bh')[1]
test_res.to_csv(os.path.join(dir_plotting, 'F3G_boxes_speck_stats.csv'), index=False)

----

In [22]:
## Figure 3 H&I - UMAP reduction of LL- and SL- retained introns coloured by Speckle enrichment and GC content

## scatter plot coloured by nuclear Speckle association

tab_hl_rev_plot = tab_hl_rev[tab_hl_rev.class_hls.isin(['LL-RIs', 'SL-RIs'])].copy()

plot_scatter_colored(
    tab_data=tab_hl_rev_plot,
    x_name='UMAP1_HLrev', y_name='UMAP2_HLrev',
    color_col="Speckle_Enrichment", vmin=-2, vmax=2,
    cmap=sns.cubehelix_palette(as_cmap=True, rot=.4, start=-.05),
    point_size=14, fig_size=(6.5,5.25),
    show_plot=False, file_name=os.path.join(dir_plotting, 'F3H_umap_hlrev_speckle_enrichment.svg')
)

plot_scatter_colored(
    tab_data=tab_hl_rev_plot,
    x_name='UMAP1_HLrev', y_name='UMAP2_HLrev',
    color_col="GC_Content", vmin=0, vmax=80,
    cmap=sns.cubehelix_palette(as_cmap=True, rot=.4, start=-.05),
    point_size=12, fig_size=(6.5,5.25),
    show_plot=False, file_name=os.path.join(dir_plotting, 'F3I_umap_hlrev_gc_content.svg')
)

# Source data export for Figure 3H&I
tab_hl_rev_plot.to_csv(os.path.join(dir_plotting, 'F3HI_umap_hlrev_source.csv'), index=True)

---

In [ ]:
## Figure 3 J - Boxplots showing the distribution of MFE in IR and HL classes

tab_hls_ss = pd.read_csv(os.path.join(dir_data, 'MFE_estimates_hls_all.csv'), index_col=0).merge(
    tab_meta[['LENGTH', 'stability', 'hl_gated_intwise_scaled_logged']],
    left_index=True, right_index=True, how='inner'
)
tab_hls_ss['MFE_lnorm'] = tab_hls_ss['MFE'] / tab_hls_ss['LENGTH']
tab_hls_ss['class_hls'] = pd.Categorical(tab_hls_ss['class_hls'], categories=['LL-RIs', 'SL-RIs'], ordered=True)
## --------------------------------
binder = []
a = tab_hls_ss.loc[tab_hls_ss['class_hls'] == 'LL-RIs', 'MFE_lnorm'].dropna()
b = tab_hls_ss.loc[tab_hls_ss['class_hls'] == 'SL-RIs', 'MFE_lnorm'].dropna()
n_ll, n_sl = len(a), len(b)
stat_t, p_t = ttest_ind(a, b, equal_var=False)
test_res_hls = pd.DataFrame(
    {
        'comparison': 'LL-RIs vs SL-RIs',
        'group_X': 'LL-RIs',
        'group_Y': 'SL-RIs',
        'n_X': n_ll,
        'n_Y': n_sl,
        'n_total': n_ll + n_sl,
        'value': 'MFE [(kJ/Mol) / Length]',
        'group': '-',
        'test': ["Welch's t-test"],
        'statistic': [stat_t],
        'p_value': [p_t]
    }
)

gg_mfe_hls = ggplot(tab_hls_ss) +\
    geom_boxplot(
        aes(x='class_hls', y='MFE_lnorm', fill='class_hls', alpha='class_hls'),
        width=.7, outlier_size=0.7, outlier_alpha=0.35
    ) +\
    labs(y='MFE [(kJ/Mol) / Length]', x='IR HL group') +\
    scale_fill_manual(values={'SL-RIs': '#a76794', 'LL-RIs': '#2a1f3f'}) +\
    scale_alpha_manual(values={'SL-RIs': .7, 'LL-RIs': .7}) +\
    coord_cartesian(xlim=[-0.5, 1.5]) +\
    ggsize(200, 650) +\
    theme_settings +\
    theme(legend_position='none')
ggsave(gg_mfe_hls, filename='F3J_boxes_MFE_hls.svg', path=dir_plotting)

# Source data export for Figure 3J (HL classes)
tab_hls_ss.to_csv(os.path.join(dir_plotting, 'F3J_boxes_MFE_hls_source.csv'))

## ---------------------------------
test_res = pd.concat([test_res_hls], axis=0)
test_res['p_adj'] = multipletests(test_res['p_value'], method='fdr_bh')[1]
test_res.to_csv(os.path.join(dir_plotting, 'F3J_boxes_MFE_hls_stats.csv'), index=False)

## ---------------------------------------------------------------------------------------------------
## ---------------------------------------------------------------------------------------------------
# IR
# Figure 3 I - Boxplots showing the distribution of MFE in IR and HL classes
tab_ir_ss = pd.read_csv(os.path.join(dir_data, 'MFE_estimates_ir.csv'), index_col=0).merge(
    tab_meta[['LENGTH', 'stability', 'hl_gated_intwise_scaled_logged']],
    left_index=True, right_index=True, how='inner'
)
tab_ir_ss['MFE_lnorm'] = tab_ir_ss['MFE'] / tab_ir_ss['LENGTH']
## --------------------------------
binder = []
a = tab_ir_ss.loc[tab_ir_ss['class_ir'] == 'Retained', 'MFE_lnorm'].dropna()
b = tab_ir_ss.loc[tab_ir_ss['class_ir'] == 'Spliced', 'MFE_lnorm'].dropna()
n_ret, n_spl = len(a), len(b)
stat_t, p_t = ttest_ind(a, b, equal_var=False)
test_res_ir = pd.DataFrame(
    {
        'comparison': 'Retained vs Spliced',
        'group_X': 'Retained',
        'group_Y': 'Spliced',
        'n_X': n_ret,
        'n_Y': n_spl,
        'n_total': n_ret + n_spl,
        'value': 'MFE [(kJ/Mol) / Length]',
        'group': '-',
        'test': ["Welch's t-test"],
        'statistic': [stat_t],
        'p_value': [p_t]
    }
)

gg_mfe_ir = ggplot(tab_ir_ss) +\
    geom_boxplot(
        aes(x='class_ir', y='MFE_lnorm', fill='class_ir', alpha='class_ir'),
        width=.7, outlier_size=0.7, outlier_alpha=0.35
    ) +\
    labs(y='MFE [(kJ/Mol) / Length]', x='IR group') +\
    scale_fill_manual(values={'Spliced': '#a76794', 'Retained': '#2a1f3f'}) +\
    scale_alpha_manual(values={'Spliced': .7, 'Retained': .7}) +\
    coord_cartesian(xlim=[-0.5, 1.5]) +\
    ggsize(200, 650) +\
    theme_settings +\
    theme(legend_position='none')
ggsave(gg_mfe_ir, filename='F3I_boxes_MFE_ir.svg', path=dir_plotting)

# Source data export for Figure 3I (IR classes)
tab_ir_ss.to_csv(os.path.join(dir_plotting, 'F3I_boxes_MFE_ir_source.csv'))

## ---------------------------------
test_res = pd.concat([test_res_ir], axis=0)
test_res['p_adj'] = multipletests(test_res['p_value'], method='fdr_bh')[1]
test_res.to_csv(os.path.join(dir_plotting, 'F3I_boxes_MFE_ir_stats.csv'), index=False)

---

In [ ]:
## Figure 3 K - Dot plots showing enrichment of G-quad motifs in IR classes

### 
tmp_ir_gquad = tab_meta.dropna(subset=['PIR_Nuc_baseline']).copy()
class_conds = [
    tmp_ir_gquad['PIR_Nuc_baseline'] >= 30,
    tmp_ir_gquad['PIR_Nuc_baseline'] < 1
]
class_names = ['Retained', 'Spliced']
tmp_ir_gquad['class_ir'] = np.select(class_conds, class_names, default='else')
tmp_ir_gquad = tmp_ir_gquad[tmp_ir_gquad.class_ir != 'else']
tmp_ir_gquad = tmp_ir_gquad.reset_index()
##
seq_to_class = {}
id_idx = tmp_ir_gquad.columns.get_loc('EVENT') + 1
class_idx = tmp_ir_gquad.columns.get_loc('class_ir') + 1
for rowtuple in tmp_ir_gquad.itertuples():
    seq_to_class[rowtuple[id_idx]] = rowtuple[class_idx] # append class
##
seq_dict = load_fasta_to_dict(
    fasta_path=os.path.join(dir_data, 'all_vastdb_introns_50fks.fasta'),
    ids_keep=tmp_ir_gquad.EVENT.to_list(),
    id_func=lambda x: x
)
compiled = compile_patterns(motifs, overlap=True)
df_feat = summarize_sequences(seq_dict=seq_dict, group_map=seq_to_class, compiled_patterns=compiled)
df_feat['count_lnorm'] = 1000 * df_feat['count'] / df_feat['length']
df_feat['count_lnorm_logged'] = np.log1p(df_feat['count_lnorm'])
##
result = fit_glm_enrichment(df_feat, family="poisson")
result['inverse_lgrr'] = np.log2(1 / result['rate_ratio'])
result['inverse_lgci_high'] = np.log2(1 / result['ci_low'])
result['inverse_lgci_low'] = np.log2(1 / result['ci_high'])
##
gg_GqaudEnrcih_ir = ggplot(result) +\
    geom_point(aes(x='motif', y='inverse_lgrr')) +\
    geom_errorbar(aes(x='motif', ymin='inverse_lgci_low', ymax='inverse_lgci_high')) +\
    geom_hline(yintercept=0, color='red') +\
    labs(y='G-quad motif hits\nlog2FC(Retained - Spliced)', x='G-quad motif') +\
    coord_cartesian(ylim=[0, 2.4]) +\
    theme_settings +\
    ggsize(250, 500)
ggsave(gg_GqaudEnrcih_ir, filename='F3K_dots_GquadEnrich_ir.svg', path=dir_plotting)

# Source data export for Supplementary Fig3F
result.to_csv(os.path.join(dir_plotting, 'F3K_dots_GquadEnrich_ir_source.csv'), index=False)

result.to_csv(os.path.join(dir_plotting, 'F3K_dots_GquadEnrich_ir_stats.csv'), index=False)

---

In [35]:
## Supplementary Fig3 L - meta-profiles of G-quad density in IR classes

## G-quad metaprofiles in IR HL groups
hits = scan_regex_hits(seq_dict, compiled, group_map=seq_to_class)
meta_df, meta_info = metacoverage_regex(
    hits_df=hits,
    seq_lookup=seq_dict,
    group_col='group',
    label_col='motif',
    flank_len=50,
    nbins_left=20,
    nbins_core=100,
    nbins_right=20,
    per_seq_mode='binary_fraction'
)
##
meta_df_2t = meta_df[meta_df.motif.isin(['G4_2t_short', 'G4_2t_long'])].copy()
gg_MetaGquad2T_ir = ggplot(meta_df_2t) +\
    geom_line(aes('bin', 'coverage', color='group', alpha='group'), size=1.5) +\
    scale_color_manual(values={'Spliced': '#a76794', 'Retained': '#2a1f3f'}) +\
    scale_alpha_manual(values={'Spliced': .7, 'Retained': .7}) +\
    geom_vline(xintercept=20) +\
    geom_vline(xintercept=120) +\
    facet_wrap('motif', nrow=1) +\
    labs(y='G-quad motif profile', x='Intron body bin', color='IR group:', alpha='IR group:') +\
    theme_settings +\
    ggsize(1200, 350)
ggsave(gg_MetaGquad2T_ir, filename='F3L_metaprofile_Gquad2T_ir.svg', path=dir_plotting)
meta_df_2t.to_csv(os.path.join(dir_plotting, 'F3L_metaprofile_Gquad2T_ir_source.csv'), index=False)

##
meta_df_3t = meta_df[meta_df.motif.isin(['G4_3t_short', 'G4_3t_long'])].copy()
gg_MetaGquad3T_ir = ggplot(meta_df_3t) +\
    geom_line(aes('bin', 'coverage', color='group', alpha='group'), size=1.5) +\
    scale_color_manual(values={'Spliced': '#a76794', 'Retained': '#2a1f3f'}) +\
    scale_alpha_manual(values={'Spliced': .7, 'Retained': .7}) +\
    geom_vline(xintercept=20) +\
    geom_vline(xintercept=120) +\
    facet_wrap('motif', nrow=1) +\
    labs(y='G-quad motif profile', x='Intron body bin', color='IR group:', alpha='IR group:') +\
    theme_settings +\
    ggsize(1200, 350)
ggsave(gg_MetaGquad3T_ir, filename='F3L_metaprofile_Gquad3T_ir.svg', path=dir_plotting)
meta_df_3t.to_csv(os.path.join(dir_plotting, 'F3L_metaprofile_Gquad3T_ir_source.csv'), index=False)